In [ ]:
# 学習
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model_path = 'models/pre_trained/yolov5s.pt'
model = Model(config_path)
model.load_state_dict(torch.load(model_path)['model'].state_dict())  # yolov5s.ptは、学習済みの重みファイル
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

hyp_path = "data/hyps/hyp.scratch-low.yaml"
with open(hyp_path, errors="ignore") as f:
    hyp = yaml.safe_load(f)

model.hyp = hyp

# カスタム損失関数
criterion = CustomLoss(model)

# オプティマイザ
optimizer = optim.Adam(model.parameters(), lr=0.001)
# scaler = torch.cuda.amp.GradScaler(enabled=True) # 高速化ライブラリ必要であれば利用したい

# データローダ
img_dir = './data/coco128/images/train2017'
annotation_dir = './data/coco128/labels/train2017'
train_dataset = CustomDataset(
    img_dir=img_dir,
    annotation_dir=annotation_dir,
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)

# tqdmの表示フォーマット
TQDM_BAR_FORMAT = '{l_bar}{bar:10}{r_bar}'

# トレーニングループ
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, total=len(train_loader), bar_format=TQDM_BAR_FORMAT)
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss, loss_items = criterion(outputs, targets)
        # scaler.scale(loss).backward()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        pbar.set_description(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    

# トレーニング済みモデルの保存
torch.save(model.state_dict(), 'models/fine_tuned/yolov5s_finetuned.pth')
print("model save!")


In [1]:
import cv2
import numpy as np
import os
from pathlib import Path
import glob

IMG_FORMATS = "bmp", "dng", "jpeg", "jpg", "mpo", "png", "tif", "tiff", "webp", "pfm"  # include image suffixes

def letterbox(im, new_shape=(640, 640), color=(114, 114, 114), auto=True, scaleFill=False, scaleup=True, stride=32):
    """Resizes and pads an image to a new shape with optional scaling, filling, and stride-multiple constraints."""
    shape = im.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    if not scaleup:  # only scale down, do not scale up (for better val mAP)
        r = min(r, 1.0)

    # Compute padding
    ratio = r, r  # width, height ratios
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding
    if auto:  # minimum rectangle
        dw, dh = np.mod(dw, stride), np.mod(dh, stride)  # wh padding
    elif scaleFill:  # stretch
        dw, dh = 0.0, 0.0
        new_unpad = (new_shape[1], new_shape[0])
        ratio = new_shape[1] / shape[1], new_shape[0] / shape[0]  # width, height ratios

    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    im = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)  # add border
    return im, ratio, (dw, dh)

class LoadImages:
    # YOLOv3 image/video dataloader, i.e. `python detect.py --source image.jpg/vid.mp4`
    def __init__(self, path, img_size=640, stride=32, auto=True, transforms=None):
        """Initializes the data loader for YOLOv3, supporting image, video, directory, and '*.txt' path lists with
        customizable image sizing.
        """
        if isinstance(path, str) and Path(path).suffix == ".txt":  # *.txt file with img/vid/dir on each line
            path = Path(path).read_text().rsplit()
        files = []
        for p in sorted(path) if isinstance(path, (list, tuple)) else [path]:
            p = str(Path(p).resolve())
            if "*" in p:
                files.extend(sorted(glob.glob(p, recursive=True)))  # glob
            elif os.path.isdir(p):
                files.extend(sorted(glob.glob(os.path.join(p, "*.*"))))  # dir
            elif os.path.isfile(p):
                files.append(p)  # files
            else:
                raise FileNotFoundError(f"{p} does not exist")

        images = [x for x in files if x.split(".")[-1].lower() in IMG_FORMATS]
        ni = len(images)

        self.img_size = img_size
        self.stride = stride
        self.files = images
        self.nf = ni  # number of files
        self.mode = "image"
        self.auto = auto
        self.transforms = transforms  # optional
        assert self.nf > 0, (
            f"No images or videos found in {p}. "
            f"Supported formats are:\nimages: {IMG_FORMATS}"
        )

    def __iter__(self):
        """Initializes the iterator by resetting count to zero and returning the iterator instance itself."""
        self.count = 0
        return self

    def __next__(self):
        """Advances to the next file in the dataset, raising StopIteration when all files are processed."""
        if self.count == self.nf:
            raise StopIteration
        path = self.files[self.count]

        # Read image
        self.count += 1
        im0 = cv2.imread(path)  # BGR
        assert im0 is not None, f"Image Not Found {path}"
        s = f"image {self.count}/{self.nf} {path}: "

        if self.transforms:
            im = self.transforms(im0)  # transforms
        else:
            im = letterbox(im0, self.img_size, stride=self.stride, auto=self.auto)[0]  # padded resize
            im = im.transpose((2, 0, 1))[::-1]  # HWC to CHW, BGR to RGB
            im = np.ascontiguousarray(im)  # contiguous

        return path, im, im0, s

    def __len__(self):
        """Returns the number of files in the dataset."""
        return self.nf  # number of files

In [4]:
# 推論
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

from torchvision import transforms
from PIL import Image

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model_path = 'models/fine_tuned/yolov5s_finetuned.pth'
model = Model(config_path)
model.load_state_dict(torch.load(model_path))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 推論モード
model.eval()

image_path = "data/images/bus.jpg"
# image = Image.open(image_path).convert("RGB")
# transform = transforms.Compose([
#             transforms.Resize((576, 576)),
#             transforms.ToTensor(),
#         ])
dataset = LoadImages(image_path)
for path, im, im0s, s in dataset:
    im = torch.from_numpy(im).to(model.device)
    im = im.half() if model.fp16 else im.float()  # uint8 to fp16/32
    im /= 255  # 0 - 255 to 0.0 - 1.0
    if len(im.shape) == 3:
        im = im[None]  # expand for batch dim
    # Inference
    visualize = increment_path(save_dir / Path(path).stem, mkdir=True) if visualize else False
    pred = model(im, augment=augment, visualize=visualize)

    # NMS
    pred = non_max_suppression(pred, conf_thres, iou_thres, classes, agnostic_nms, max_det=max_det)


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]              
  8                -1  1   1182720  

YOLOv3s summary: 214 layers, 7235389 parameters, 7235389 gradients, 16.6 GFLOPs



3


TypeError: conv2d() received an invalid combination of arguments - got (list, Parameter, NoneType, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, tuple of ints padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!list!, !Parameter!, !NoneType!, !tuple!, !tuple!, !tuple!, int)
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, str padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!list!, !Parameter!, !NoneType!, !tuple!, !tuple!, !tuple!, int)
